In [21]:
from mp_api.client import MPRester
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np

load_dotenv()
API_KEY = os.getenv("MP_API_KEY")

In [28]:
# known perovskite-forming B-site elements
PEROVSKITE_B_SITES = {
    "Ti", "Zr", "Hf",   # group 4
    "V", "Nb", "Ta",    # group 5
    "Cr", "Mo", "W",    # group 6
    "Mn", "Fe", "Co",   # group 7-9
    "Ni", "Cu",         # group 10-11
    "Al", "Ga", "In",   # group 13
    "Sn", "Pb",         # group 14
    "Bi",               # group 15
}

with MPRester(API_KEY) as mpr:
    results = mpr.materials.summary.search(
        formula="**O3",
        fields=[
            "material_id",
            "formula_pretty",
            "band_gap",
            "formation_energy_per_atom",
            "energy_above_hull",
            "volume",
            "density",
            "elements",
            "nelements",
        ]
    )

df_oxide = pd.DataFrame([r.model_dump() for r in results])

# keep only 3-element compounds (A + B + O)
df_oxide = df_oxide[df_oxide["nelements"] == 3]

# keep only rows where at least one B-site element is present
def has_b_site(elements):
    return any(str(e) in PEROVSKITE_B_SITES for e in elements)

df_oxide = df_oxide[df_oxide["elements"].apply(has_b_site)]

print(f"Oxide perovskites after filtering: {len(df_oxide)}")
print(df_oxide["formula_pretty"].sample(min(15, len(df_oxide))).tolist())

Retrieving SummaryDoc documents: 100%|██████████| 2547/2547 [00:00<00:00, 3697.87it/s]

Oxide perovskites after filtering: 1455
['ZnCoO3', 'FeBiO3', 'MnPbO3', 'AlInO3', 'EuGaO3', 'TiCdO3', 'CuCO3', 'AlVO3', 'TmAlO3', 'DyTiO3', 'SrMoO3', 'NiBiO3', 'AlCoO3', 'PuPbO3', 'NaVO3']


In [29]:
with MPRester(API_KEY) as mpr:
    results_halide = mpr.materials.summary.search(
        formula="**X3",          # X = halide placeholder
        chemsys=["Cs-Pb-I", "Cs-Pb-Br", "Cs-Pb-Cl", "Cs-Sn-I", "Cs-Sn-Br", "Cs-Sn-Cl",
                 "Rb-Pb-I", "Rb-Pb-Br", "K-Pb-I"],  # halide perovskites
        fields=[
            "material_id",
            "formula_pretty",
            "band_gap",
            "formation_energy_per_atom",
            "energy_above_hull",
            "volume",
            "density",
            "elements",
            "nelements",
        ]
    )
df_halide = pd.DataFrame([r.model_dump() for r in results_halide])
print(f"Halide perovskites pulled: {len(df_halide)}")
df_halide.head()

Retrieving SummaryDoc documents: 100%|██████████| 24/24 [00:00<00:00, 588674.25it/s]

Halide perovskites pulled: 24


,elements,nelements,formula_pretty,volume,density,material_id,formation_energy_per_atom,energy_above_hull,band_gap,fields_not_requested
0,"[Br, Cs, Sn]",3,CsSnBr3,783.020931,4.167799,mp-cszuj,-1.572084,0.000000,0.9682,"[builder_meta, nsites, composition, compositio..."
1,"[Br, Cs, Sn]",3,CsSnBr3,199.074979,4.098297,mp-bogs,-1.560889,0.011195,0.5984,"[builder_meta, nsites, composition, compositio..."
2,"[Cl, Cs, Sn]",3,CsSnCl3,172.323849,3.449497,mp-cixkh,-1.554896,0.022687,0.9787,"[builder_meta, nsites, composition, compositio..."
3,"[Cl, Cs, Sn]",3,CsSnCl3,707.754903,3.359528,mp-bonq,-1.577583,0.000000,2.9154,"[builder_meta, nsites, composition, compositio..."
4,"[Br, Pb, Rb]",3,RbPbBr3,214.225808,4.126662,mp-ceupk,-1.573023,0.039393,1.7508,"[builder_meta, nsites, composition, compositio..."


In [30]:
df_oxide["family"] = "oxide"
df_halide["family"] = "halide"
df = pd.concat([df_oxide, df_halide], ignore_index=True)

df = df[df["energy_above_hull"] < 0.1]  # filter for stability
df = df[df["band_gap"] > 0]  # filter for non-metals
df = df[df["band_gap"].notna()]  # filter for entries with band gap data

df = df.reset_index(drop=True)
print(f"Final dataset size after filtering: {len(df)}")
print(df[["formula_pretty", "band_gap", "family"]].head(10))

Final dataset size after filtering: 603
  formula_pretty  band_gap family
0           VHO3    0.0094  oxide
1         CoNiO3    0.7645  oxide
2         LaCoO3    0.6739  oxide
3          VCoO3    1.1514  oxide
4          NbHO3    2.6221  oxide
5         TiNiO3    1.4963  oxide
6           VHO3    2.1077  oxide
7          NbHO3    2.5310  oxide
8          NbHO3    0.0005  oxide
9           VHO3    0.6363  oxide


In [31]:
print(f"Total rows: {len(df)}")
print(f"\nFamily breakdown:")
print(df["family"].value_counts())
print(f"\nBand gap range:")
print(df["band_gap"].describe())
print(f"\nSample formulas:")
print(df["formula_pretty"].sample(15).tolist())

Total rows: 603

Family breakdown:
family
oxide     579
halide     24
Name: count, dtype: int64

Band gap range:
count    603.000000
mean       2.036453
std        1.222286
min        0.000200
25%        1.080150
50%        1.996200
75%        2.901850
max        6.091400
Name: band_gap, dtype: float64

Sample formulas:
['ErCrO3', 'NaNbO3', 'CaWO3', 'CdSnO3', 'CaZrO3', 'LaFeO3', 'AlBiO3', 'YBiO3', 'YGaO3', 'PrGaO3', 'CoNiO3', 'CsPbBr3', 'LaCoO3', 'LiNbO3', 'SrCrO3']


In [32]:
df.to_csv("data/perovskite_dataset.csv", index=False)
print("Saved.")

Saved.
